# 02. 업력 문장 골라내기

## 무엇을 하려는 건가

공고문에는 이런 문장들이 섞여 있다.

```
"창업 후 7년 이내 중소기업"              ← 업력 조건이 맞다
"장기재직(3년 이상)이 필요한 근로자"      ← 근속연수다. 업력이 아니다
"3개월 이상 영업 중인 소상공인"           ← 개월이다. 년으로 읽으면 안 된다
"만 19세 ~ 만 45세"                      ← 사람 나이다
```

**이 넷을 구분하는 것**이 목표다. 지금은 정규식 규칙이 하고 있는데,
규칙은 모양이 조금만 달라지면 놓친다.

여기서는 문장 332개를 보여주며 **기계가 스스로 구분법을 배우게** 한다.

---

## 먼저 알아둘 것 — 이 실험의 한계

정답지를 사람이 아니라 **기존 규칙이 만들었다.** 그래서 이대로 점수를 재면
"규칙을 얼마나 잘 흉내 내나"를 재는 것이지 "규칙보다 나은가"가 아니다.

그래서 두 단계로 간다.

| 단계 | 하는 일 | 알 수 있는 것 |
| --- | --- | --- |
| 1단계 | 규칙이 만든 정답지로 학습·채점 | 제대로 배웠는지 (점검용) |
| 2단계 | 규칙과 모델의 **판단이 갈린 것만** 사람이 확인 | 누가 맞는지 (진짜 결론) |

2단계가 핵심이다. 둘의 의견이 같은 건 볼 필요가 없고,
**갈린 것만 보면 되니까 확인할 양이 확 줄어든다.**

In [1]:
# -*- coding: utf-8 -*-
import io, os, sys, json, collections, random

# 01번과 같은 이유로 작업 폴더를 data-collection 으로 옮긴다.
ROOT = os.path.abspath('..')
if os.path.basename(os.getcwd()) == 'ml':
    os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print('작업 폴더 :', os.getcwd())
DATA   = os.path.join(ROOT, 'ml', 'data')
MODELS = os.path.join(ROOT, 'ml', 'models')
os.makedirs(MODELS, exist_ok=True)

LABELS = os.path.join(DATA, 'age_labels.jsonl')
assert os.path.exists(LABELS), '01_dataset.ipynb 를 먼저 돌려야 한다'

rows = [json.loads(l) for l in io.open(LABELS, encoding='utf-8') if l.strip()]
print('문장 %d개' % len(rows))
print('  업력 맞음 (1) :', sum(1 for r in rows if r['label'] == 1))
print('  업력 아님 (0) :', sum(1 for r in rows if r['label'] == 0))

작업 폴더 : C:\SKN-TEST\SKN32-FINAL-1TEAM\data-collection
문장 332개
  업력 맞음 (1) : 130
  업력 아님 (0) : 202


---

## 1. 공부용과 시험용 나누기

여기서도 똑같다. **공부한 문장으로 시험 보면 안 된다.**

다만 이번엔 문장 하나가 곧 문제 하나라서, 앞 노트북처럼 복잡하게 묶을 필요가 없다.
그냥 섞어서 자르되, **업력 맞음/아님 비율을 양쪽에 똑같이** 맞춘다.
한쪽에 쏠리면 점수가 왜곡되기 때문이다.

`SEED` 는 섞을 때 쓴 주사위 눈금이다. 적어두면 몇 번을 돌려도 같게 나뉜다.

In [2]:
from sklearn.model_selection import train_test_split

SEED = 20260916

texts  = [r['quote'] for r in rows]
labels = [r['label'] for r in rows]

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    texts, labels, range(len(rows)),
    test_size=0.25,        # 4개 중 1개를 시험용으로
    stratify=labels,       # 업력 맞음/아님 비율을 양쪽에 맞춘다
    random_state=SEED,
)

print('공부용 %d개 (업력 %d · 아님 %d)' % (
    len(X_train), sum(y_train), len(y_train) - sum(y_train)))
print('시험용 %d개 (업력 %d · 아님 %d)' % (
    len(X_test), sum(y_test), len(y_test) - sum(y_test)))

공부용 249개 (업력 97 · 아님 152)
시험용 83개 (업력 33 · 아님 50)


---

## 2. 어떻게 가르치나

두 단계다.

**① 문장을 숫자로 바꾼다 (TF-IDF)**

기계는 글자를 못 읽으니 숫자로 바꿔야 한다. 방법은 단순하다 —
**글자 조각이 몇 번 나오는지 세는 것**이다.

```
"창업 후 7년 이내"  →  '창업'  '업 '  ' 후'  '7년'  '년 '  ' 이'  '이내' …
```

한국어는 띄어쓰기가 들쭉날쭉해서 단어로 자르면 놓치는 게 많다.
그래서 **글자 2~4개씩 잘라서** 센다. '창업'과 '창업후'를 둘 다 잡을 수 있다.

이름의 IDF 부분은 "아무 문장에나 다 나오는 조각은 덜 중요하게 친다"는 뜻이다.

**② 선을 긋는다 (로지스틱 회귀)**

숫자로 바뀐 문장들을 늘어놓고 **업력인 것과 아닌 것 사이에 선을 하나 긋는다.**
새 문장이 오면 선의 어느 쪽에 떨어지는지 보고 판단한다.

이게 전부다. 신경망도 GPU도 필요 없고 몇 초면 끝난다.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

model = make_pipeline(
    TfidfVectorizer(
        analyzer='char_wb',      # 글자 단위로 자른다 (한국어에 유리)
        ngram_range=(2, 4),      # 2~4글자씩
        min_df=2,                # 딱 한 번만 나온 조각은 버린다 (우연일 가능성)
        sublinear_tf=True,
    ),
    LogisticRegression(
        max_iter=2000,
        class_weight='balanced', # 업력 아님이 더 많으므로 균형을 맞춘다
        C=1.0,
    ),
)

model.fit(X_train, y_train)
print('학습 끝')

학습 끝


---

## 3. 시험 — 얼마나 맞히나

점수를 볼 때 **정확도 하나만 보면 안 된다.** 업력 아님이 더 많아서,
전부 "업력 아님"이라고만 찍어도 정확도가 61%나 나오기 때문이다.

그래서 두 가지를 같이 본다.

| 이름 | 뜻 | 낮으면 생기는 일 |
| --- | --- | --- |
| **정밀도** | 업력이라고 한 것 중 진짜 업력인 비율 | 엉뚱한 걸 업력으로 읽어 **잘못된 자격 판정** |
| **재현율** | 진짜 업력 중 찾아낸 비율 | 업력을 놓쳐서 **판정을 못 함** |

이 프로젝트에서는 **정밀도가 더 중요하다.** 놓치는 건 "확인 필요"로 남기면 되지만,
잘못 읽으면 사용자에게 틀린 자격 판정을 보여주게 된다.

from sklearn.metrics import classification_report, confusion_matrix

pred = model.predict(X_test)

print(classification_report(
    y_test, pred,
    target_names=['업력 아님(0)', '업력 맞음(1)'],
    digits=3, zero_division=0))

In [5]:
import numpy as np
cm = confusion_matrix(y_test, pred)

print('                    모델 판단')
print('                업력아님   업력맞음')
print('실제 업력아님      %4d      %4d' % (cm[0][0], cm[0][1]))
print('실제 업력맞음      %4d      %4d' % (cm[1][0], cm[1][1]))
print()
print('대각선(%d, %d)이 맞힌 것이다.' % (cm[0][0], cm[1][1]))
print('오른쪽 위 %d개 = 업력이 아닌데 업력이라고 했다  ← 이게 제일 위험하다' % cm[0][1])
print('왼쪽 아래 %d개 = 업력인데 놓쳤다' % cm[1][0])

                    모델 판단
                업력아님   업력맞음
실제 업력아님        49         1
실제 업력맞음         4        29

대각선(49, 29)이 맞힌 것이다.
오른쪽 위 1개 = 업력이 아닌데 업력이라고 했다  ← 이게 제일 위험하다
왼쪽 아래 4개 = 업력인데 놓쳤다


### 틀린 것을 눈으로 본다

숫자만 보면 왜 틀렸는지 모른다. 실제 문장을 봐야 고칠 방향이 보인다.

In [6]:
proba = model.predict_proba(X_test)[:, 1]   # 업력일 확률

wrong = [(X_test[i], y_test[i], pred[i], proba[i])
         for i in range(len(X_test)) if y_test[i] != pred[i]]

print('틀린 것 %d개\n' % len(wrong))
for text, truth, got, p in sorted(wrong, key=lambda x: -abs(x[3] - 0.5))[:12]:
    mark = '업력 아닌데 업력이라 함' if truth == 0 else '업력인데 놓침'
    print('[%s · 확신 %.0f%%]' % (mark, max(p, 1 - p) * 100))
    print('   ', text[:80])
    print()

틀린 것 5개

[업력인데 놓침 · 확신 72%]
    업력이 2년 이상이며 최근 6개월 또는 전년 대비 매출 감소로 경영 위기에 직면한 소상공인

[업력 아닌데 업력이라 함 · 확신 67%]
    7년 이내 초기창업 기업 또는 예비창업자(선정 시, 사업자 등록 必)

[업력인데 놓침 · 확신 59%]
    주관연구개발기관 및 공동연구개발기관 中 국내기업은 접수마감일 현재 사업자등록증 및 법인등기부등본 기준으로 창업 1년 이상 경과하고, 한국산업기술

[업력인데 놓침 · 확신 58%]
    증축 및 개보수의 경우, 공고일 기준으로 1년 전*부터 영업 중인 업체 * 관광사업 등록증 기준 1년 이상 관광사업자로 영위하여야 함.

[업력인데 놓침 · 확신 53%]
    중진공 지정 부실징후기업 또는 업력 5년 초과 기업 중 다음에 해당하는 한계기업 - 2년 연속 적자기업 중 자기자본 전액 잠식 기업 ⑭ - 3년



---

## 4. 애매하면 판단하지 않는다

이 프로젝트의 원칙은 **"모르는 것과 아닌 것은 다르다"** 이다.
`gate.py` 가 통과·미달·**확인 필요** 세 값을 쓰는 것도 같은 이유다.

모델도 똑같이 만든다. 확률이 애매한 구간(예: 35~65%)은
0도 1도 아닌 **"모르겠음"** 으로 빼낸다.

그러면 판단하는 건수는 줄지만, **판단한 것의 정확도는 올라간다.**
아래에서 그 맞바꿈이 실제로 얼마나 되는지 본다.

In [7]:
def with_abstain(low, high):
    """확률이 low~high 사이면 판단을 보류한다."""
    judged = [(t, p) for t, p in zip(y_test, proba) if not (low <= p <= high)]
    if not judged:
        return 0, 0.0
    ok = sum(1 for t, p in judged if t == (1 if p > high else 0))
    return len(judged), ok / len(judged)

print('%-16s %-12s %s' % ('보류 구간', '판단한 건수', '판단한 것의 정확도'))
print('-' * 52)
for low, high in ((0.5, 0.5), (0.4, 0.6), (0.35, 0.65), (0.3, 0.7), (0.25, 0.75)):
    n, acc = with_abstain(low, high)
    label = '없음(전부 판단)' if low == high else '%.2f ~ %.2f' % (low, high)
    print('%-16s %-12s %.1f%%' % (label, '%d / %d' % (n, len(y_test)), acc * 100))

보류 구간            판단한 건수       판단한 것의 정확도
----------------------------------------------------
없음(전부 판단)        83 / 83      94.0%
0.40 ~ 0.60      58 / 83      96.6%
0.35 ~ 0.65      45 / 83      95.6%
0.30 ~ 0.70      31 / 83      96.8%
0.25 ~ 0.75      16 / 83      100.0%


---

## 5. 진짜 시험 — 규칙과 모델이 갈린 것

여기부터가 핵심이다.

지금까지 잰 점수는 **규칙이 만든 정답지 기준**이라, 잘 나와봐야
"규칙을 잘 흉내 냈다"는 뜻이다. 규칙보다 나은지는 아직 모른다.

**둘의 판단이 갈린 문장만 뽑아서 사람이 보면 된다.**
같은 판단을 한 건 볼 필요가 없으니, 확인할 양이 크게 줄어든다.

아래 칸이 그 목록을 파일로 뽑는다.

In [8]:
# 전체 332개에 대해 모델 판단을 구하고, 규칙(=기존 라벨)과 갈린 것만 모은다.
all_proba = model.predict_proba(texts)[:, 1]

disagree = []
for i, r in enumerate(rows):
    rule = r['label']
    mine = 1 if all_proba[i] > 0.5 else 0
    if rule != mine:
        disagree.append({
            'notice_id': r['notice_id'],
            'quote': r['quote'],
            'rule_says': rule,
            'model_says': mine,
            'model_confidence': round(float(max(all_proba[i], 1 - all_proba[i])), 3),
            'human': None,        # ← 사람이 채울 칸. 1 = 업력 / 0 = 아님
        })

disagree.sort(key=lambda d: -d['model_confidence'])

out = os.path.join(DATA, 'age_disagreements.jsonl')
with io.open(out, 'w', encoding='utf-8') as f:
    for d in disagree:
        f.write(json.dumps(d, ensure_ascii=False) + '\n')

print('전체 %d개 중 판단이 갈린 것 %d개 (%.1f%%)'
      % (len(rows), len(disagree), len(disagree) / len(rows) * 100))
print('→', out)
print()
print('이 파일의 human 칸을 채우면 규칙과 모델 중 누가 맞았는지 알 수 있다.')

전체 332개 중 판단이 갈린 것 18개 (5.4%)
→ C:\SKN-TEST\SKN32-FINAL-1TEAM\data-collection\ml\data\age_disagreements.jsonl

이 파일의 human 칸을 채우면 규칙과 모델 중 누가 맞았는지 알 수 있다.


In [9]:
# 갈린 것 중 모델이 특히 확신하는 것부터 본다. 여기에 규칙의 허점이 드러난다.
for d in disagree[:15]:
    print('규칙:%s  모델:%s (확신 %.0f%%)' % (
        '업력' if d['rule_says'] else '아님',
        '업력' if d['model_says'] else '아님',
        d['model_confidence'] * 100))
    print('   ', d['quote'][:80])
    print()

규칙:업력  모델:아님 (확신 72%)
    업력이 2년 이상이며 최근 6개월 또는 전년 대비 매출 감소로 경영 위기에 직면한 소상공인

규칙:아님  모델:업력 (확신 67%)
    7년 이내 초기창업 기업 또는 예비창업자(선정 시, 사업자 등록 必)

규칙:아님  모델:업력 (확신 65%)
    예비창업자(팀) 또는 7년 이내 창업기업

규칙:아님  모델:업력 (확신 64%)
    충남 소재 7년 미만 기술기반 창업기업

규칙:아님  모델:업력 (확신 64%)
    공고일 기준 1인 (예비)창업기업 또는 7년 이내 창업자

규칙:업력  모델:아님 (확신 59%)
    주관연구개발기관 및 공동연구개발기관 中 국내기업은 접수마감일 현재 사업자등록증 및 법인등기부등본 기준으로 창업 1년 이상 경과하고, 한국산업기술

규칙:업력  모델:아님 (확신 58%)
    증축 및 개보수의 경우, 공고일 기준으로 1년 전*부터 영업 중인 업체 * 관광사업 등록증 기준 1년 이상 관광사업자로 영위하여야 함.

규칙:업력  모델:아님 (확신 57%)
    과제2(제조업): 설립 10년 이상 노후 시설을 보유한 상시 근로자 30인미만 화재․폭발․누출․질식 위험 보유 영세 사업장(60개사)

규칙:업력  모델:아님 (확신 55%)
    - 설립 후 3년 이상 경과하였거나, 국가공인시험분석기관(KOLAS인증포함)으로 인증지원 등을 주업으로 하는 회사ㆍ기관 패키지 지원 - 설립 후

규칙:아님  모델:업력 (확신 55%)
    7년 미만 창업기업(신산업 분야 기업은 10년 이내까지 인정)으로,

규칙:업력  모델:아님 (확신 54%)
    ② 공고일 기준 관내 점포를 임차하여 영업 중인 청년으로 관할 세무서에 사업자등록을 마친 뒤 5년이 경과하지 않은 소상공인

규칙:업력  모델:아님 (확신 53%)
    중진공 지정 부실징후기업 또는 업력 5년 초과 기업 중 다음에 해당하는 한계기업 - 2년 연속 적자기업 중 자기자본 전액 잠식 기업 ⑭ - 3년

규칙:업력  모

---

## 6. 모델 저장

제출물이 되는 파일이다. 몇 백 KB 정도라 git 에 올려도 된다.

In [10]:
import joblib

path = os.path.join(MODELS, 'age_classifier_v1.joblib')
joblib.dump(model, path)
print('저장 →', path)
print('크기 : %.1f KB' % (os.path.getsize(path) / 1024))

# 잘 불러와지는지 확인
loaded = joblib.load(path)
for s in ['창업 후 5년 이내 중소기업',
          '장기재직 3년 이상 근로자',
          '만 20세 이상 39세 이하',
          '6개월 이상 영업 중인 소상공인']:
    p = loaded.predict_proba([s])[0][1]
    print('  %-32s → 업력일 확률 %.0f%%' % (s, p * 100))

저장 → C:\SKN-TEST\SKN32-FINAL-1TEAM\data-collection\ml\models\age_classifier_v1.joblib
크기 : 75.3 KB
  창업 후 5년 이내 중소기업                  → 업력일 확률 77%
  장기재직 3년 이상 근로자                   → 업력일 확률 27%
  만 20세 이상 39세 이하                  → 업력일 확률 30%
  6개월 이상 영업 중인 소상공인                → 업력일 확률 21%


---

## 정리

만들어진 것

- `ml/models/age_classifier_v1.joblib` — 학습한 모델 (제출물)
- `ml/data/age_disagreements.jsonl` — 규칙과 갈린 목록 (사람 확인용)

**다음에 할 일**

`age_disagreements.jsonl` 의 `human` 칸을 채운다. 1 = 업력 / 0 = 아님.
다 채울 필요는 없고 위에서부터 100개 정도면 충분하다.

그걸로 이런 표를 만들 수 있고, 이게 결과서의 결론이 된다.

| | 규칙이 맞음 | 모델이 맞음 |
| --- | --- | --- |
| 갈린 100건 중 | ?건 | ?건 |

모델이 더 많이 맞으면 규칙을 교체할 근거가 되고,
규칙이 더 많이 맞으면 **"학습해봤으나 규칙이 나아서 쓰지 않기로 했다"** 가 된다.
**어느 쪽이든 결과서에 쓸 수 있는 결론이다.** 실험은 좋게 나와야만 가치가 있는 게 아니다.